# Hit-and-run sampler on the transportation polytope

Scratch notebook for testing `sample_hit_run` (`cntmosaic/predict/_polytope_sampler.py`) and plotting its output.

In [ ]:
import jax

# Use 64-bit floats so margin errors stay at machine precision over long chains.
jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import numpy as np
import pandas as pd
import altair as alt
from scipy import stats
from numpyro.diagnostics import effective_sample_size

from cntmosaic.predict import (
    check_margins,
    diagonal_is_feasible,
    sample_hit_run,
)
from cntmosaic.predict._polytope_sampler import _chord, independence_table

alt.data_transformers.disable_max_rows()

## Validation: $K = 2$

With $\mathbf{r} = (3, 1)$ and $\mathbf{c} = (2, 2)$ the whole table is fixed by $x_{11}$:

$$
\mathbf{X} = \begin{pmatrix} x_{11} & 3 - x_{11} \\ 2 - x_{11} & x_{11} - 1 \end{pmatrix},
$$

and non-negativity gives $1 \le x_{11} \le 2$. The uniform distribution on $\mathcal{T}(\mathbf{r}, \mathbf{c})$ is therefore $x_{11} \sim \mathrm{Uniform}(1, 2)$ exactly.

The polytope is a line segment, so every chord spans the whole segment and each hit-and-run step is an independent uniform draw. No burn-in or thinning is needed, and a Kolmogorov–Smirnov (KS) test against $\mathrm{Uniform}(1, 2)$ is valid.

In [ ]:
r2 = jnp.array([3.0, 1.0])
c2 = jnp.array([2.0, 2.0])
exact = stats.uniform(loc=1.0, scale=1.0)

n_draws = 1_000
x11 = sample_hit_run(
    jax.random.key(0), r2, c2, num_samples=n_draws, num_warmup=0
)[:, 0, 0]
x11 = np.asarray(x11)

ks = stats.kstest(x11, exact.cdf)
print(f"range of x11: [{x11.min():.4f}, {x11.max():.4f}]")
print(f"KS statistic = {ks.statistic:.4f}, p = {ks.pvalue:.3f}")

A single KS p-value says little on its own. We repeat the test over 300 independent replications (one chain each, run as a batch). If the sampler is correct, the p-values are $\mathrm{Uniform}(0, 1)$ and the test rejects at the 5% level in about 5% of replications.

In [ ]:
n_reps = 300
x11_reps = sample_hit_run(
    jax.random.key(1),
    jnp.tile(r2, (n_reps, 1)),
    jnp.tile(c2, (n_reps, 1)),
    num_samples=n_draws,
    num_warmup=0,
)[..., 0, 0]

p_values = np.array([stats.kstest(x, exact.cdf).pvalue for x in np.asarray(x11_reps)])
print(f"rejection rate at 5%: {np.mean(p_values < 0.05):.3f}")
print(
    f"KS test of p-values against Uniform(0, 1): p = {stats.kstest(p_values, 'uniform').pvalue:.3f}"
)

In [ ]:
counts, edges = np.histogram(x11, bins=20, range=(1.0, 2.0), density=True)
hist_df = pd.DataFrame({"start": edges[:-1], "end": edges[1:], "density": counts})

hist = (
    alt.Chart(hist_df)
    .mark_bar(opacity=0.7)
    .encode(
        x=alt.X("start:Q", title="x₁₁", scale=alt.Scale(domain=[0.9, 2.1])),
        x2="end:Q",
        y=alt.Y("density:Q", title="Density"),
    )
)
exact_density = (
    alt.Chart(
        pd.DataFrame(
            {"x": [0.9, 1.0, 1.0, 2.0, 2.0, 2.1], "density": [0, 0, 1, 1, 0, 0]}
        )
    )
    .mark_line(color="black")
    .encode(x="x:Q", y="density:Q")
)
left = (hist + exact_density).properties(
    title=f"x₁₁ from {n_draws} draws vs Uniform(1, 2)", width=300, height=250
)

ecdf_df = pd.DataFrame(
    {
        "p": np.sort(p_values),
        "ecdf": np.arange(1, n_reps + 1) / n_reps,
    }
)
ecdf = (
    alt.Chart(ecdf_df)
    .mark_line(interpolate="step-after")
    .encode(
        x=alt.X("p:Q", title="KS p-value"),
        y=alt.Y("ecdf:Q", title="ECDF"),
    )
)
diagonal = (
    alt.Chart(pd.DataFrame({"p": [0, 1], "ecdf": [0, 1]}))
    .mark_line(color="black", strokeDash=[4, 4])
    .encode(x="p:Q", y="ecdf:Q")
)
right = (ecdf + diagonal).properties(
    title=f"KS p-values over {n_reps} replications", width=300, height=250
)

left | right

## Validation: Case A (equal margins), $K = 10$

Take $\mathbf{r} = \mathbf{c} = \mathbf{1}$ with $K = 10$, so $\dim \mathcal{T} = (K - 1)^2 = 81$. The polytope is unchanged by any permutation of rows or columns, so under the uniform distribution every cell has the same marginal law and

$$
\mathbb{E}[x_{ij}] = \frac{1}{K}
$$

exactly. This gives an exact test in a dimension where hit-and-run steps are no longer independent.

We follow the supplement: 128 chains started at the independence table $\mathbf{X}_0 = \mathbf{r}\mathbf{c}^\top / N$, $2 \times 10^4$ burn-in steps, then $10^5$ steps keeping every 250th table (400 per chain, 51 200 in total). Successive kept tables are still correlated, so we compute the standard error of each cell mean from the spread of the 128 chain means, which are independent.

In [ ]:
K = 10
n_chains = 128
num_warmup, n_steps, thin = 20_000, 100_000, 250

r_a = jnp.ones(K)
c_a = jnp.ones(K)

xs_a = sample_hit_run(
    jax.random.key(2),
    jnp.tile(r_a, (n_chains, 1)),
    jnp.tile(c_a, (n_chains, 1)),
    num_samples=n_steps // thin,
    num_warmup=num_warmup,
    thin=thin,
)
xs_a = np.asarray(xs_a)  # (chains, draws, K, K)
xs_a.shape

In [ ]:
def summarise(xs, x_ref):
    """Summary statistics of stored tables against a reference table."""
    r, c = xs.sum(-1)[0, 0], xs.sum(-2)[0, 0]
    chain_means = xs.mean(1)
    x_bar = chain_means.mean(0)
    se = chain_means.std(0, ddof=1) / np.sqrt(xs.shape[0])
    ess = effective_sample_size(xs)
    return (
        pd.Series(
            {
                "max margin error": max(
                    np.abs(xs.sum(-1) - r).max(), np.abs(xs.sum(-2) - c).max()
                ),
                "smallest entry": xs.min(),
                "max |x̄ - x_ref| / se": (np.abs(x_bar - x_ref) / se).max(),
                "max |x̄ - x_ref| / x_ref": (np.abs(x_bar - x_ref) / x_ref).max(),
                "ESS min": ess.min(),
                "ESS median": np.median(ess),
            }
        ),
        x_bar,
        se,
    )


x_exact_a = np.full((K, K), 1 / K)
summary_a, x_bar_a, se_a = summarise(xs_a, x_exact_a)
summary_a

If the sampler is correct, the 100 $z$-scores $(\bar{x}_{ij} - 1/K) / \mathrm{se}_{ij}$ are roughly standard normal, so the largest in absolute value should be around 2.5–3. By symmetry, the empirical CDFs (ECDFs) of any cells must also coincide.

In [ ]:
z_df = pd.DataFrame(
    {
        "row": np.repeat(np.arange(1, K + 1), K),
        "col": np.tile(np.arange(1, K + 1), K),
        "z": ((x_bar_a - x_exact_a) / se_a).ravel(),
    }
)
z_max = float(np.ceil(np.abs(z_df["z"]).max()))
z_map = (
    alt.Chart(z_df)
    .mark_rect()
    .encode(
        x=alt.X("col:O", title="Column j"),
        y=alt.Y("row:O", title="Row i"),
        color=alt.Color(
            "z:Q",
            title="z-score",
            scale=alt.Scale(scheme="redblue", domain=[-z_max, z_max], reverse=True),
        ),
        tooltip=["row", "col", alt.Tooltip("z:Q", format=".2f")],
    )
    .properties(title="z-scores of cell means against 1/K", width=250, height=250)
)

cells = [(0, 0), (0, K - 1), (K // 2, K // 2 - 1), (K - 1, K - 1)]
grid = np.linspace(0, np.quantile(xs_a, 0.999), 200)
ecdf_cells = pd.concat(
    [
        pd.DataFrame(
            {
                "x": grid,
                "ecdf": np.searchsorted(
                    np.sort(xs_a[..., i, j].ravel()), grid, side="right"
                )
                / xs_a[..., i, j].size,
                "cell": f"x_({i + 1},{j + 1})",
            }
        )
        for i, j in cells
    ]
)
ecdf_chart = (
    alt.Chart(ecdf_cells)
    .mark_line()
    .encode(
        x=alt.X("x:Q", title="Cell value"),
        y=alt.Y("ecdf:Q", title="ECDF"),
        color=alt.Color("cell:N", title="Cell"),
        strokeDash="cell:N",
    )
    .properties(title="ECDFs of four cells", width=300, height=250)
)

(z_map | ecdf_chart).resolve_scale(color="independent")

## Scale of the margins

In the application the margins range from about $10^4$ to several million. Hit-and-run is scale-equivariant: if $\mathbf{X}$ is uniform on $\mathcal{T}(\mathbf{r}, \mathbf{c})$, then $s\mathbf{X}$ is uniform on $\mathcal{T}(s\mathbf{r}, s\mathbf{c})$. The sampler therefore runs on the margins divided by their total $N$ and multiplies the tables by $N$ at the end.

We use the Case B margins, $N = 100$, $r_i \propto i$ and $c_j \propto 0.75^{j-1}$, and run them again with the margins multiplied by $s = 2^{17}$, so $N \approx 1.3 \times 10^7$, using the same key. Multiplying by a power of 2 is exact in floating point, so the normalised margins are bit-for-bit identical and the two runs must agree exactly after rescaling.

For any other scale the normalised margins differ in the last bit. Hit-and-run is chaotic, so after a few hundred steps the two paths then separate completely, although both remain valid draws from the uniform distribution.

In [ ]:
i = np.arange(1, K + 1)
r_b = 100 * i / i.sum()
c_b = 100 * 0.75 ** (i - 1) / (0.75 ** (i - 1)).sum()

scale = 2.0**17
x_small = sample_hit_run(jax.random.key(3), r_b, c_b, num_samples=200)
x_large = sample_hit_run(
    jax.random.key(3), scale * r_b, scale * c_b, num_samples=200
)

rel_diff = jnp.abs(x_large - scale * x_small).max() / x_large.max()
margin_err = jnp.abs(x_large.sum(-1) - scale * r_b).max() / (scale * r_b).max()
print(f"max relative difference after rescaling: {rel_diff:.1e}")
print(f"max relative margin error at N = 1.3e7: {margin_err:.1e}")

## How many steps are needed from $\mathbf{X}_0$?

In the application each pair of margins needs a single table, so there is no need for long chains or thinning. Each pair gets its own chain, run for $T$ steps from $\mathbf{X}_0 = \mathbf{r}\mathbf{c}^\top / N$, and the last table is kept. The question is how large $T$ must be for that table to be a draw from the uniform distribution.

We mimic this with 1000 independent chains per $T$ and compare the 1000 tables at step $T$ with a reference sample. The reference is 1000 further chains run for $5 \times 10^4$ steps. Using independent chains, rather than thinned draws from a few chains, makes the reference draws independent, so the comparisons below are valid. We track three quantities:

1. **Bias:** the largest, over cells, of $|\bar{x}_{ij}(T) - \bar{x}^{\mathrm{ref}}_{ij}| / \mathrm{sd}^{\mathrm{ref}}_{ij}$.
2. **Distribution distance:** the largest, over cells, of the two-sample Kolmogorov–Smirnov (KS) statistic between the draws of $x_{ij}$ at step $T$ and the reference draws.
3. **Distance from start:** the median of $\|\mathbf{X}_T - \mathbf{X}_0\|_F$, divided by the same median for the reference. It is 0 at the start and tends to 1.

For 1 and 2 the dashed lines are the 5% critical values, Bonferroni-corrected over the $K^2$ cells. Once a curve stays below its line, the draws at step $T$ cannot be told apart from the reference with this many chains.

In [ ]:
def burn_in_curve(r, c, steps, key, n_chains=1_000, ref_steps=50_000):
    """Compare tables after T steps from X_0 with a long-run reference, for each T in steps."""
    r, c = check_margins(r, c)
    tile = lambda m: jnp.tile(m, (n_chains, 1))

    def draw(key, num_steps):
        x = sample_hit_run(
            key, tile(r), tile(c), num_samples=1, num_warmup=num_steps - 1
        )
        return np.asarray(x[:, 0])

    key_ref, key_run = jax.random.split(key)
    x0 = np.asarray(independence_table(r, c))
    ref = draw(key_ref, ref_steps)
    ref_mean, ref_sd = ref.mean(0), ref.std(0)
    ref_dist = np.median(np.linalg.norm(ref - x0, axis=(1, 2)))

    rows = []
    for T, k in zip(steps, jax.random.split(key_run, len(steps))):
        x = np.broadcast_to(x0, ref.shape) if T == 0 else draw(k, T)
        rows.append(
            {
                "steps": T,
                "bias": np.nanmax(np.abs(x.mean(0) - ref_mean) / ref_sd),
                "KS distance": max(
                    stats.ks_2samp(x[:, i, j], ref[:, i, j]).statistic
                    for i, j in np.ndindex(x0.shape)
                ),
                "distance from start": np.median(np.linalg.norm(x - x0, axis=(1, 2)))
                / ref_dist,
            }
        )

    alpha = 0.05 / x0.size
    floors = {
        "bias": stats.norm.ppf(1 - alpha / 2) * np.sqrt(2 / n_chains),
        "KS distance": np.sqrt(-np.log(alpha / 2) / 2) * np.sqrt(2 / n_chains),
        "distance from start": 1.0,
    }
    return pd.DataFrame(rows), floors

In [ ]:
steps = [0, 100, 400, 1_600, 6_400, 25_600]

curves, floors = {}, None
for name, (r, c), seed in [("Case A", (r_a, c_a), 4), ("Case B", (r_b, c_b), 5)]:
    curves[name], floors = burn_in_curve(r, c, steps, jax.random.key(seed))

curve_df = pd.concat([df.assign(case=name) for name, df in curves.items()])
curve_df.round(3)

In [ ]:
def metric_panel(metric):
    base = alt.Chart(curve_df).encode(
        x=alt.X("steps:Q", title="Steps T from X₀", scale=alt.Scale(type="symlog")),
    )
    lines = base.mark_line(point=True).encode(
        y=alt.Y(f"{metric}:Q", title=metric),
        color=alt.Color("case:N", title=None),
    )
    floor = (
        alt.Chart(pd.DataFrame({"y": [floors[metric]]}))
        .mark_rule(color="black", strokeDash=[4, 4])
        .encode(y="y:Q")
    )
    return (lines + floor).properties(title=metric.capitalize(), width=220, height=220)


alt.hconcat(*[metric_panel(m) for m in floors])

### Conclusion

|                          | $T$ where bias and KS distance reach the noise floor | Distance from start at $T = 25\,600$ |
|--------------------------|------------------------------------------------------|--------------------------------------|
| Case A (equal margins)   | about 1 600                                          | 1.00                                 |
| Case B (unequal margins) | above 25 600 (likely about $5 \times 10^4$)          | 0.89                                 |

- The number of steps needed depends strongly on how unequal the margins are. With equal margins about $1\,600$ steps suffice. With the Case B margins (ratios of about 10 within $\mathbf{r}$ and 13 within $\mathbf{c}$), even $25\,600$ steps fall just short.
- The bias at $T = 0$ is zero in Case A only because $\mathbf{X}_0$ is exactly the uniform mean there. The KS distance and the distance from start show that the chain still needs time to spread out.
- The reference itself is run for $5 \times 10^4$ steps, so for Case B it may not be fully converged either. The Case B threshold is therefore a lower bound.
- Before choosing `num_warmup` for the application, run `burn_in_curve` on representative real margins, for example the most unequal pair across posterior draws and age pairs, and take about twice the $T$ at which the curves reach the noise floor.

## Fixed diagonal

In the contact setting the diagonal cell $x_{kk}$ counts contacts within stratum $k$, so fixing it states how assortative the mixing is. Let $S$ be the strata whose diagonal is fixed, $x_{kk} = \delta_k$ for $k \in S$, with $m = |S|$. The sampler then draws uniformly from

$$
\mathcal{T}_S(\mathbf{r}, \mathbf{c}, \boldsymbol{\delta}) = \{\mathbf{X} \in \mathcal{T}(\mathbf{r}, \mathbf{c}) : x_{kk} = \delta_k,\ k \in S\},
$$

whose dimension is $(K - 1)^2 - m$. Directions are projected onto the tables with zero row and column sums and a zero diagonal on $S$, and the chain starts at $\operatorname{diag}(\boldsymbol{\delta})$ plus the quasi-independence table of the remaining cells (Algorithm 2 of the supplement).

### Exact check: $K = 3$ with the whole diagonal fixed

With $K = 3$ and $S = \{1, 2, 3\}$ the dimension is $(3 - 1)^2 - 3 = 1$: the tables form a line segment along $\boldsymbol{\Pi} - \boldsymbol{\Pi}^\top$, where $\boldsymbol{\Pi}$ is the cyclic permutation matrix. Every chord is the whole segment, so each hit-and-run step is an independent uniform draw on it. We take $\mathbf{r} = \mathbf{c} = \mathbf{1}$ and $\boldsymbol{\delta} = (0.2, 0.3, 0.4)$, and compare $x_{12}$ with the uniform distribution on its range.

In [ ]:
r3 = jnp.ones(3)
delta3 = jnp.array([0.2, 0.3, 0.4])
print("strictly feasible:", bool(diagonal_is_feasible(r3, r3, delta3)))

x3 = sample_hit_run(
    jax.random.key(20), r3, r3, num_samples=5_000, num_warmup=0, diagonal=delta3
)

# Ends of the segment, from any one draw
cycle = jnp.roll(jnp.eye(3), 1, axis=1)
t_lo, t_hi = _chord(x3[0], cycle - cycle.T)
lo, hi = float(x3[0, 0, 1] + t_lo), float(x3[0, 0, 1] + t_hi)

x12 = np.asarray(x3[:, 0, 1])
ks3 = stats.kstest(x12, stats.uniform(lo, hi - lo).cdf)
print(f"x12 lies in [{lo:.3f}, {hi:.3f}]")
print(f"max diagonal error: {jnp.abs(jnp.diagonal(x3, axis1=1, axis2=2) - delta3).max():.1e}")
print(f"max margin error: {jnp.abs(x3.sum(-1) - 1).max():.1e}")
print(f"KS statistic = {ks3.statistic:.4f}, p = {ks3.pvalue:.3f}")

In [ ]:
counts3, edges3 = np.histogram(x12, bins=25, range=(lo, hi), density=True)
hist3 = alt.Chart(
    pd.DataFrame({"start": edges3[:-1], "end": edges3[1:], "density": counts3})
).mark_bar(opacity=0.7).encode(
    x=alt.X("start:Q", title="x₁₂"),
    x2="end:Q",
    y=alt.Y("density:Q", title="Density"),
)
uniform3 = alt.Chart(
    pd.DataFrame({"x": [lo, hi], "density": [1 / (hi - lo)] * 2})
).mark_line(color="black").encode(x="x:Q", y="density:Q")
(hist3 + uniform3).properties(
    title="x₁₂ with the whole diagonal fixed vs the uniform density", width=350, height=250
)

### How much smaller is the set?

Fixing the diagonal lowers the dimension by $m$, but more importantly it removes mass from the free cells: every table in $\mathcal{T}_S$ differs from any other by at most $2N'$ in $\ell_1$ distance, where $N' = N - \sum_{k \in S} \delta_k$ (Corollary 1 of the supplement).

We use the Case B margins and fix the whole diagonal at $\delta_k = \rho\, r_k c_k / N$ with $\rho = 1.5$, i.e. 50% more within-stratum contact than under proportionate mixing (Remark 3 of the supplement). Both samplers run 128 chains with $2 \times 10^4$ burn-in steps, keeping 200 tables per chain every 50 steps.

In [ ]:
rho = 1.5
delta_b = rho * r_b * c_b / r_b.sum()
print("strictly feasible:", bool(diagonal_is_feasible(r_b, c_b, delta_b)))
print(f"N = {r_b.sum():.1f}, N' = {r_b.sum() - delta_b.sum():.1f}")

run_kwargs = dict(num_samples=200, num_warmup=20_000, thin=50)
tile = lambda m: jnp.tile(m, (128, 1))
xs_free = np.asarray(
    sample_hit_run(jax.random.key(21), tile(r_b), tile(c_b), **run_kwargs)
).reshape(-1, K, K)
xs_fixed = np.asarray(
    sample_hit_run(
        jax.random.key(22), tile(r_b), tile(c_b), diagonal=delta_b, **run_kwargs
    )
).reshape(-1, K, K)

print(f"max diagonal error: {np.abs(np.diagonal(xs_fixed, axis1=1, axis2=2) - delta_b).max():.1e}")
print(f"max margin error: {np.abs(xs_fixed.sum(-1) - r_b).max():.1e}")


def l1_spread(xs):
    """Mean l1 distance of the tables from their average."""
    return np.abs(xs - xs.mean(0)).sum(axis=(1, 2)).mean()


print(f"mean l1 distance from the average table: free {l1_spread(xs_free):.2f}, "
      f"fixed diagonal {l1_spread(xs_fixed):.2f}")

In [ ]:
sd_ratio = xs_fixed.std(0) / xs_free.std(0)
off_diagonal = ~np.eye(K, dtype=bool)
print(f"sd of off-diagonal cells, fixed / free: median {np.median(sd_ratio[off_diagonal]):.2f}, "
      f"range [{sd_ratio[off_diagonal].min():.2f}, {sd_ratio[off_diagonal].max():.2f}]")

sd_df = pd.DataFrame({
    "row": np.repeat(np.arange(1, K + 1), K),
    "col": np.tile(np.arange(1, K + 1), K),
    "ratio": np.where(off_diagonal, sd_ratio, np.nan).ravel(),
})
sd_map = alt.Chart(sd_df.dropna()).mark_rect().encode(
    x=alt.X("col:O", title="Column j"),
    y=alt.Y("row:O", title="Row i"),
    color=alt.Color("ratio:Q", title="sd ratio", scale=alt.Scale(scheme="blues", reverse=True)),
    tooltip=["row", "col", alt.Tooltip("ratio:Q", format=".2f")],
).properties(title="sd of each cell, fixed / free diagonal", width=250, height=250)

cells_b = [(0, 1), (4, 6), (9, 0)]
ecdf_b = []
for name, xs in [("free", xs_free), ("fixed diagonal", xs_fixed)]:
    for i, j in cells_b:
        values = np.sort(xs[:, i, j])
        ecdf_b.append(pd.DataFrame({
            "x": values[::50],
            "ecdf": (np.arange(1, values.size + 1) / values.size)[::50],
            "cell": f"x_({i + 1},{j + 1})",
            "sampler": name,
        }))
ecdf_b_chart = alt.Chart(pd.concat(ecdf_b)).mark_line().encode(
    x=alt.X("x:Q", title="Cell value"),
    y=alt.Y("ecdf:Q", title="ECDF"),
    color=alt.Color("cell:N", title="Cell"),
    strokeDash=alt.StrokeDash("sampler:N", title="Sampler"),
).properties(title="ECDFs of three off-diagonal cells", width=300, height=250)

(sd_map | ecdf_b_chart).resolve_scale(color="independent")